# 🛡️ GARDA-JKN | Enterprise Advanced Data Science Pipeline
**End-to-End BPJS Claim Anomaly Detection (FKRTL + Diagnosis Sekunder)**
Standards: INA-CBG Standard 2024-2025, Healthkathon BPJS 2026.

Methodology Architecture:
1. **Adaptive Ingestion**: Automatic resolution of BPJS 2025 Stata files (`202403_fkrtl.dta` & `202405_diagnosissekunder.dta`) with memory-efficient batching and synthetic fallback.
2. **Clinical Feature Engineering (R2)**: Fixes baseline column bugs (LOS from `FKL04-FKL03`, Severity from `FKL19`/`FKL23`, Care Setting from `FKL10`, Base CBG from `FKL19`) and extracts suspect comorbidity codes (`E43`, `J96`, `Sepsis/Shock`, `AKI`) and 4 Clinical Incoherence rules.
3. **Stratified Isolation Forest (R1)**: Grouped outlier detection by `(Base CBG, Kelas RS)` with 4-level hierarchical cascading fallback for small strata ($N < 50$) and continuous calibrated anomaly scores in [0, 1].
4. **Balanced Bagging XGBoost (R3)**: Ensemble of parallel XGBoost classifiers trained on 100% positive fraud claims paired with independent balanced 1:1 negative subsamples (`scale_pos_weight=1.0`). Strictly ZERO SMOTE/ADASYN.
5. **Operational Evaluation & TreeSHAP Auditor Reason Codes (R4)**: PR-AUC (Average Precision), Precision@K, and translation of ensemble TreeSHAP feature impacts into human-readable Indonesian BPJS Auditor Reason Codes (`ARC-CLIN-01`, `ARC-UPCODE-E43`, `ARC-COST-01`, etc.).
6. **Production Serialization**: Export certified artifacts for Lapis 2 & 3 orchestration.

In [ ]:
# 1. INSTALL & IMPORT LIBRARIES
import sys
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Dependencies import
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    precision_recall_curve,
    auc,
    average_precision_score,
)
import xgboost as xgb
import shap

# Safe Headless Plotting Guard (matplotlib/seaborn)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_MATPLOTLIB = True
    sns.set_theme(style="whitegrid")
    plt.rcParams['figure.figsize'] = (10, 6)
    print("✅ Environment Data Science & Visualisasi siap!")
except ImportError:
    HAS_MATPLOTLIB = False
    print("ℹ️ [Headless Environment] matplotlib/seaborn tidak terpasang. Visualisasi dialihkan ke ringkasan teks.")


### 2. DATA LOADING (Memuat Data BPJS 2025)
Mendukung resolusi path dinamis: membaca file lokal Stata (`202403_fkrtl.dta` dan `202405_diagnosissekunder.dta`), Google Colab Drive, atau fallback data sintetis jika file belum dimount.

In [ ]:
# Resolusi Path Fleksibel & Pemuatan Data
from garda_ds_pipeline import BPJSDataLoader, DEFAULT_FKRTL_PATHS, DEFAULT_SEKUNDER_PATHS

print("🔍 Memeriksa ketersediaan dataset BPJS Kesehatan Rilis 2025...")
df_fkrtl, df_sek = BPJSDataLoader.load_data(n_samples=50000, random_state=42)

print(f"✅ Data FKRTL dimuat       : {df_fkrtl.shape[0]:,} baris, {df_fkrtl.shape[1]} kolom")
print(f"✅ Data Diagnosis Sekunder  : {df_sek.shape[0]:,} baris, {df_sek.shape[1]} kolom")


### 3. PREPROCESSING & CLINICAL FEATURE ENGINEERING (R2)
Memperbaiki bug kolom baseline:
1. **Length of Stay (LOS)**: Dihitung dari selisih `FKL04` (Tanggal Pulang) - `FKL03` (Tanggal Datang). (Bukan `FKL25` yang merupakan kode provinsi).
2. **Severity Level**: Diambil dari digit ke-4 / suffix `FKL19` (`-0`, `-I`, `-II`, `-III`) atau `FKL23`. (Bukan `FKL08` yang merupakan jenis faskes).
3. **Care Setting**: Diambil dari `FKL10` (`RJTL` vs `RITL`). (Bukan `FKL07` yang merupakan kepemilikan RS).
4. **Hospital Class (Kelas RS)**: Diambil dari `FKL09` (`RS Kelas A/B/C/D`).
5. **Base CBG**: Diekstrak dari `FKL19.str.rsplit('-', 1)[0]`.
6. **Ekstraksi Komorbiditas Kritis**: ICD-10 `E43` (Malnutrisi Berat), `J96` (Gagal Napas), `A40/A41/R57` (Sepsis/Syok), `N17` (Gagal Ginjal Akut).
7. **Clinical Incoherence Rules**:
   - Sepsis/Syok dengan 0 hari ICU/ventilator dan rawat singkat $\le 2$ hari pulang Sehat / Outpatient.
   - J96 Gagal Napas tanpa ventilator dan rawat singkat $\le 2$ hari.
   - Severity Level 3 tanpa ada diagnosis sekunder penyerta.
   - Phantom Inpatient Billing (LOS = 0 pada RITL dengan biaya tinggi dan non-operatif).

In [ ]:
from garda_ds_pipeline import ClinicalFeatureExtractor

print("🔄 Mengekstrak fitur klinis dan aturan Clinical Incoherence...")
extractor = ClinicalFeatureExtractor()
df_feat = extractor.transform(df_fkrtl, df_sek)

print(f"✅ Feature Engineering selesai: {df_feat.shape[0]:,} klaim diproses.")
print(f"   • Kasus Suspect Malnutrisi Berat (E43) : {df_feat['FLAG_SUSPECT_E43'].sum():,} klaim")
print(f"   • Kasus Suspect Gagal Napas (J96)      : {df_feat['FLAG_SUSPECT_J96'].sum():,} klaim")
print(f"   • Kasus Suspect Sepsis / Syok          : {df_feat['FLAG_SUSPECT_SEPSIS'].sum():,} klaim")
print(f"   • Kasus Suspect Gagal Ginjal Akut (N17): {df_feat['FLAG_SUSPECT_AKI'].sum():,} klaim")
print(f"   • Total Pelanggaran Clinical Incoherence: {df_feat['CLINICAL_INCOHERENCE'].sum():,} klaim")
print(f"     - Sepsis/Syok tanpa ICU & LOS <= 2 hr: {df_feat['INCOH_SEPSIS_NO_ICU'].sum():,}")
print(f"     - J96 tanpa Bantuan Ventilator       : {df_feat['INCOH_J96_NO_VENT'].sum():,}")
print(f"     - Severity 3 tanpa Diagnosis Sekunder: {df_feat['INCOH_UPCODING_SEV3'].sum():,}")
print(f"     - Phantom Daycare Non-Operatif RITL  : {df_feat['INCOH_PHANTOM_DAYCARE'].sum():,}")


### 4. EXPLORATORY DATA ANALYSIS (EDA)
Menganalisis pola distribusi biaya, lama rawat inap, dan keparahan klaim BPJS Kesehatan. Prosedur didesain agar kompatibel baik pada lingkungan GUI maupun lingkungan headless/server.

In [ ]:
# Analisis Distribusi Finansial & Klinis
print("📊 RINGKASAN DISTRIBUSI FINANSIAL & KLINIS:")
summary_df = df_feat.groupby('SEVERITY_LEVEL')[['LOS_HARI', 'BIAYA_TAGIH', 'BIAYA_PER_HARI', 'JML_DIAG_SEKUNDER']].agg(['median', 'mean']).round(2)
print(summary_df)

if HAS_MATPLOTLIB:
    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    sns.boxplot(x='SEVERITY_LEVEL', y='BIAYA_TAGIH', data=df_feat[df_feat['BIAYA_TAGIH'] < 30_000_000])
    plt.title('Distribusi Biaya Tagih vs Severity INA-CBG')
    plt.xlabel('Severity Level (0=RJTL, 1=Ringan, 2=Sedang, 3=Berat)')
    plt.ylabel('Biaya (Rupiah)')

    plt.subplot(1, 2, 2)
    sns.barplot(x='SEVERITY_LEVEL', y='JML_DIAG_SEKUNDER', data=df_feat)
    plt.title('Rata-Rata Komorbiditas Berdasarkan Severity')
    plt.xlabel('Severity Level')
    plt.ylabel('Rata-rata Jumlah Diagnosis Sekunder')
    plt.tight_layout()
    plt.show()
else:
    print("\n[Visualisasi Grafis]: Mode Headless aktif — ringkasan tabel numerik ditampilkan di atas.")

print("\n🔍 CLINICAL AUDIT INSIGHT:")
print("Severity 3 (Berat) memerlukan biaya intervensi tinggi dan komorbiditas signifikan.")
print("Klaim dengan Severity 3 tanpa komorbiditas atau Sepsis tanpa ICU merupakan indikasi kuat upcoding.")


### 5. STRATIFIED ISOLATION FOREST & SEMI-SUPERVISED LABELING (R1)
Mengelompokkan klaim ke dalam peer-group berdasarkan `(Base CBG, Kelas RS)` sehingga prosedur kompleks (misal transplantasi atau bedah saraf di RS Kelas A) tidak disalahartikan sebagai fraud. Menggunakan fallback hierarkis untuk strata kecil ($N < 50$) dan kalibrasi skor anomali kontinu dalam rentang $[0, 1]$.

In [ ]:
from garda_ds_pipeline import StratifiedIsolationForest, generate_semi_supervised_labels

print("⏳ Menjalankan Stratified Isolation Forest (R1) dengan Hierarchical Fallback...")
strat_iso = StratifiedIsolationForest(min_samples=50, n_estimators=60, random_state=42)
strat_iso.fit(df_feat)
df_feat['ISO_ANOMALY_SCORE'] = strat_iso.score_samples(df_feat)

print(f"✅ Anomaly Scoring Selesai. Rentang skor: [{df_feat['ISO_ANOMALY_SCORE'].min():.3f}, {df_feat['ISO_ANOMALY_SCORE'].max():.3f}]")

# Labeling Semi-Supervised: Menggabungkan Skor Anomali Kluster dengan Aturan Medis Deterministik
df_feat['FRAUD_RISK'] = generate_semi_supervised_labels(
    df_feat, df_feat['ISO_ANOMALY_SCORE'].values, contamination_threshold=0.70
)

n_fraud = df_feat['FRAUD_RISK'].sum()
print(f"🎯 Total Klaim Berisiko Anomali/Fraud: {n_fraud:,} ({n_fraud/len(df_feat)*100:.2f}% dari populasi)")


### 6. BALANCED BAGGING XGBOOST (R3)
Ansambel XGBoost paralel (5 model) yang dilatih pada 100% data positif klaim berisiko yang dipasangkan dengan subsample independen dari klaim negatif (rasio 1:1, `scale_pos_weight = 1.0`).
**DILARANG KERAS SMOTE/ADASYN**: Zero synthetic clinical data generation untuk menjamin validitas audit medis.

In [ ]:
from garda_ds_pipeline import BalancedBaggingXGBoost, ADVANCED_FEATURE_COLS

X = df_feat[ADVANCED_FEATURE_COLS]
y = df_feat['FRAUD_RISK']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Dataset Split: Train={len(X_train):,} klaim (Positif={y_train.sum():,}), Test={len(X_test):,} klaim (Positif={y_test.sum():,})")
print("🚀 Melatih BalancedBaggingXGBoost (5 Base Estimators, Rasio 1:1, scale_pos_weight=1.0)...")

bagging_xgb = BalancedBaggingXGBoost(
    n_estimators=5,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
)
bagging_xgb.fit(X_train, y_train)
print("✅ Ansambel BalancedBaggingXGBoost selesai dilatih!")


### 7. OPERATIONAL EVALUATION & SHAP AUDITOR REASON CODES (R4)
Evaluasi menggunakan metrik operasional audit klaim:
1. **PR-AUC (Precision-Recall AUC)** & **Average Precision (AP)** untuk menangani data imbalanced ekstrem.
2. **Precision@K (K=10, 50, 100)**: Evaluasi kuota harian inspeksi auditor BPJS Kesehatan.
3. **TreeSHAP Auditor Reason Codes**: Menerjemahkan nilai TreeSHAP ansambel menjadi kode alasan resmi (`ARC-CLIN-01`, `ARC-UPCODE-E43`, `ARC-COST-01`, dll.) lengkap dengan narasi klinis dan instruksi verifikasi audit.

In [ ]:
from garda_ds_pipeline import OperationalEvaluator, ShapAuditorReasonCodeGenerator

# 1. Evaluasi Operasional
y_test_proba = bagging_xgb.predict_proba(X_test)[:, 1]
y_test_pred = bagging_xgb.predict(X_test, threshold=0.5)

metrics = OperationalEvaluator.evaluate(y_test, y_test_proba, y_test_pred)
OperationalEvaluator.print_metrics_summary(metrics)

# 2. Explainable AI: TreeSHAP Auditor Reason Code Generator
reason_gen = ShapAuditorReasonCodeGenerator(bagging_xgb, ADVANCED_FEATURE_COLS)

print("\n" + "=" * 78)
print("📢 DEMONSTRASI AUDITOR REASON CODES PADA KLAIM BERISIKO TINGGI (TEST SET)")
print("=" * 78)

# Pilih sedikitnya 3 klaim berisiko tinggi dari test set
high_risk_idx = [idx for idx in X_test.index if df_feat.loc[idx, 'FRAUD_RISK'] == 1][:3]
for idx in high_risk_idx:
    row = df_feat.loc[idx]
    cid = str(row.get('FKL02', f'CLAIM-{idx}'))
    prob = float(bagging_xgb.predict_proba(pd.DataFrame([row[ADVANCED_FEATURE_COLS]]))[:, 1][0])
    explanation = reason_gen.explain_claim(row, claim_id=cid)
    print(reason_gen.format_auditor_report(explanation, risk_score=prob))


### 8. EXPORT MODEL ARTIFACTS
Menyimpan model ansambel dan metadata fitur untuk integrasi produksi dengan `app/core/ml_engine.py` dan orkestrasi LangGraph.

In [ ]:
from garda_ds_pipeline import ARTIFACTS_DIR

print(f"💾 Menyimpan artefak model ke direktori: {ARTIFACTS_DIR}...")
adv_model_p = os.path.join(ARTIFACTS_DIR, "garda_xgb_advanced.json")
adv_meta_p = os.path.join(ARTIFACTS_DIR, "garda_features_meta_advanced.json")

# Simpan estimator utama dan metadata
bagging_xgb.estimators_[0].save_model(adv_model_p)
with open(adv_meta_p, "w", encoding="utf-8") as f:
    json.dump({"feature_cols": ADVANCED_FEATURE_COLS}, f, indent=2)

print("✅ Artefak produksi berhasil disimpan:")
print(f"   • Model JSON     : {adv_model_p}")
print(f"   • Metadata JSON  : {adv_meta_p}")
print("🛡️ GARDA-JKN Advanced DS Pipeline Siap Produksi!")
